val_acc_delta < 1e-3 => 10 epoch => early stopping

|lr|optimizer|weight_decay|time(s)|epoch|gpu|train_loss|train_acc|val_loss|val_acc|val_5_acc|
|-|-|-|-|-|-|-|-|-|-|-|
|1e-3|sdg|0|29978.85973238945|1597|3080ti*4|0.5668|0.8670|0.7995|0.8018|0.9542|
|1e-3|adam|1e-6|432.56122756004333|17|3080ti*4|3.3624|0.7446|4.2530|0.6935|0.8662|
|1e-3|amsgrad|1e-6|430.4932208061218|23|3090*2|0.8104|0.8284|1.4700|0.7407|0.9139|
|1e-4|sdg|0|63100|3300 (manually stopped)|3080ti*4|1.0464|0.7973|1.2456|0.7424|0.9199|
|1e-4|adam|1e-8|1051.9337539672852|43|3080ti*4|0.5172|0.8626|0.9079|0.7791|0.9428|
|1e-4|amsgrad|1e-8|2589.3759520053864|138|3090*2|0.3189|0.9146|0.7659|0.8034|0.9580|
|1e-5|sdg|0|95700|5000 (manually stopped)|3080ti*4|2.7872|0.5605|2.9839|0.5034|0.7298|
|1e-5|adam|1e-10|7905.626512765884|357|3080ti*4|0.3469|0.9112|0.6684|0.8214|0.9644|
|1e-5|amsgrad|1e-10|6634.382847547531|553|3090*2|0.3901|0.9028|0.6679|0.8219|0.9651|

In [7]:
import json
with open("hf_class_label.json", 'r') as f:
    data = json.load(f)

with open("1e-05_amsgrad_1e-10_top5_predictions.txt", "r") as f:
    lines = f.readlines()

with open("amsgrad_pred.txt", "w") as f:
    for line in lines:
        line = line.strip().split(" ")
        line = [data[i] for i in line]
        f.write(" ".join(line) + "\n")

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '7'
import torch
import torch.nn as nn

class LinearClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(LinearClassifier, self).__init__()
        self.linear = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        return self.linear(x)
    
    def load_state_dict(self, state_dict):
        self.linear.load_state_dict(state_dict)

def compute_loss_and_accuracy(logits, labels):
    loss_fn = nn.CrossEntropyLoss()
    loss = loss_fn(logits, labels.long())
    preds = torch.argmax(logits, 1)
    accuracy = torch.mean((preds == labels).float())
    return loss, accuracy

def compute_top5_accuracy(logits, labels):
    _, top5_preds = torch.topk(logits, 5, dim=1)
    top5_accuracy = torch.mean(torch.sum(top5_preds == labels.unsqueeze(1), dim=1).float())
    return top5_accuracy

def load_data_split(data_type, indices, device):
    features_list = []
    labels_list = []
    for index in indices:
        feature_path = f'image_features/{data_type}/{data_type}_features_{index}.pt'
        features = torch.load(feature_path).to(device)
        features_list.append(features)
        if data_type != 'test':
            label_path = f'image_labels/{data_type}/{data_type}_labels_{index}.pt'
            labels = torch.load(label_path).to(device)
            labels_list.append(labels)
    return features_list, labels_list

def test(model, test_indices, devices, batch_size=1000):
    model.eval()
    num_devices = len(devices)
    test_splits = [test_indices[i::num_devices] for i in range(num_devices)]
    test_data = [load_data_split('test', split, device) for split, device in zip(test_splits, devices)]
    
    with torch.no_grad():
        for device, (test_features, _) in zip(devices, test_data):
            for inputs in test_features:
                inputs = inputs.to(device)
                logits = model(inputs).to(device)
                _, top5_preds = torch.topk(logits, 5, dim=1)
                with open(f'1e-05_sgd_0e+00_top5_predictions.txt', 'a') as f:
                    for i in range(top5_preds.size(0)):
                        line = " ".join(map(str, top5_preds[i].tolist()))
                        f.write(f'{line}\n')

input_dim = 7680
num_classes = 1000
epochs = 100000
devices = [torch.device('cuda:0')]
# model = LinearClassifier(input_dim, num_classes)
model = LinearClassifier(input_dim, num_classes)
state_dict = torch.load('model/1e-05_sgd_0e+00/linear_classifier_epoch_5000.pt')
state_dict = {k.replace('module.linear.', ''): v for k, v in state_dict.items()}
model.load_state_dict(state_dict)
model.to(devices[0])
test_indices = list(range(100))
test(model, test_indices, devices, batch_size=1000)

/tmp/ipykernel_3258221/2187328261.py:65: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load('model/1e-05_sgd_0e+00/linear_classifier_epoch_5000.pt')
/tmp/

: 